# preRA cohort scRNA analysis in Python for Monocytes 
- CertPro



In [ ]:
import h5py
import scipy.sparse as scs
import pandas as pd
import anndata
import os
import glob
from matplotlib import pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import median_abs_deviation
import scanpy as sc
# import sc_toolbox as sct
import decoupler as dc

In [ ]:
anndata.__version__

In [ ]:
# sc.settings.n_jobs = 58

In [ ]:
# define some color patterns for plotting
nejm_color = ["#BC3C29FF", "#0072B5FF", "#E18727FF", "#20854EFF", "#7876B1FF", "#6F99ADFF", "#FFDC91FF", "#EE4C97FF"]
jama_color = ["#374E55FF", "#DF8F44FF", "#00A1D5FF", "#B24745FF", "#79AF97FF", "#6A6599FF", "#80796BFF"]

In [ ]:
# define working path
data_path = '/home/jupyter/data/ra_longitudinal/reference_data/Alivernini_et_al_tissue_Macrophage/'
mtx_path = '/home/jupyter/data/ra_longitudinal/reference_data/Alivernini_et_al_tissue_Macrophage/expression_data'
fig_path = '/home/jupyter/data/ra_longitudinal/figures/mono/validation/'
meta_path = '/home/jupyter/github/ra-longitudinal/metadata/'
output_path = '/home/jupyter/data/ra_longitudinal/output_results/validation/'

# define a project name
proj_name = 'ALTRA_scRNA_mono_validation'
# sc.set_figure_params(fig_path)
sc.settings.figdir = fig_path
sc.settings.autosave=False
sc.set_figure_params(vector_friendly=True, dpi_save=300)

In [ ]:
# set fig size
plt.rcParams['figure.figsize'] = [10, 8]

# load data

In [ ]:
# load pair metadata 
pair_meta = pd.read_csv('/home/jupyter/data/ra_longitudinal/output_results/' + 'AIM3_paired_pre_post_conversion_samples.csv')
pair_meta.head()

# Analyze expression in paired comparison

In [ ]:
hp.cache_files(["b69cc4c1-3f35-4c76-8730-b6a8004004f5"])

In [ ]:
# load data
pair_mono_adata = sc.read_h5ad(
    "/home/workspace/input/569004694/UCSDCU_Y4/b69cc4c1-3f35-4c76-8730-b6a8004004f5/ALTRA_scRNA_monocytes_paired_certPro.h5ad"
)

In [ ]:
pair_mono_adata

In [ ]:
pair_mono_adata.obs['AIFI_L3_new'].value_counts()

In [ ]:
pair_mono_adata.obs['celltype_status'] = pair_mono_adata.obs['AIFI_L3_new'].astype('str') + pair_mono_adata.obs['status'].astype('str')

In [ ]:
sc.pl.dotplot(pair_mono_adata, ['TNF', 'IL1B', "IL1RN", 'CCR2', 
                           'CD14','FCGR3A'], "celltype_status", standard_scale='var',
              save=  proj_name+'_paired_samples_rna_TNF_dotpolt.png',dendrogram=True)


In [ ]:
pair_mono_adata

In [ ]:
sc.pl.dotplot(pair_mono_adata, ['TNF', 'IL1B', "IL1RN", 'CCR2', 
                           'CD14','FCGR3A'], "celltype_status", standard_scale='var',
              save=  proj_name+'_paired_samples_rna_TNF_dotpolt_status.png',dendrogram=True)

In [ ]:
# plot the markers gene list from the paper
gene_list = ['NFKBIA', 'TNF', 'IL1B', "CCL3", 'CCL4', 'ICAM1', 'FOLR2', 
            'CD14', 'FCGR3A', 'KLF6', 'NR4A1', 'DUSP1', 'ATF3']

sc.pl.dotplot(pair_mono_adata, gene_list, "AIFI_L3",  standard_scale='var', swap_axes=True,
              save=  proj_name+'_AIFI_L3_target_genes.pdf')


In [ ]:
with plt.rc_context({"figure.figsize": (8, 4)}):
    sc.pl.violin(pair_mono_adata, ["TNF"], groupby="AIFI_L3", 
                 save=  proj_name+'_rna_TNF_violin.png',
                 inner="box", rotation=90)

In [ ]:
# setting highly variable as highly deviant to use scanpy 'use_highly_variable' argument in sc.pp.pca
sc.pp.pca(pair_mono_adata, svd_solver="arpack", use_highly_variable=True)
# plot the principle component variance explained
sc.pl.pca_variance_ratio(pair_mono_adata, log=True)

In [ ]:
pair_mono_adata

In [ ]:
sc.pl.embedding(
    pair_mono_adata,
    color=['AIFI_L3_new', 'leiden_1'], legend_loc='on data',
    basis='X_tsne',
    frameon=False, ncols=2,
    save=  proj_name+'_paired_leiden_1.pdf'
)

In [ ]:
# set order for status column
pair_mono_adata.obs['status'] = pair_mono_adata.obs['status'].astype('str')
pair_mono_adata.obs.loc[pair_mono_adata.obs['status']=='pre_conv', 'status'] = 'pre-disease'
pair_mono_adata.obs['status'] = pair_mono_adata.obs['status'].astype('category').cat.reorder_categories(
    ['pre-disease', 'conversion'], ordered=True)

In [ ]:
sc.tl.embedding_density(pair_mono_adata, basis='tsne', groupby='status')

In [ ]:
sc.set_figure_params(scanpy=True, fontsize=14) 
sc.pl.embedding_density(pair_mono_adata, basis='tsne',
                        key='tsne_density_status', 
                        ncols=2, 
                        color_map= 'magma',
                       save=  proj_name+'_paired_tsne_density_status.pdf')

In [ ]:
sc.pl.embedding(
    pair_mono_adata,
    color=['status', 'leiden_1', 'AIFI_L3_new'],#legend_loc='on data',
    basis='X_tsne',
    save=  proj_name+'_paired_status.png'
)

In [ ]:
with plt.rc_context({"figure.figsize": (4, 4), "figure.dpi": (400)}):
        ax = sc.pl.embedding(
        pair_mono_adata,
        color=['AIFI_L3_new'],  #legend_loc='on data', 
        legend_fontsize='small',
        frameon=False, basis='X_tsne',
        save = proj_name+'_paired_AIFI_L3_new.pdf'
    )


In [ ]:
sc.pl.embedding(
    pair_mono_adata,
    color=['leiden_1'], legend_loc='on data',
    frameon=False, basis='X_tsne',
    save = proj_name+'_paired_leiden_1.pdf'
)

In [ ]:
sc.pl.embedding(
    pair_mono_adata,
    color=['TNF', 'IL1B', 'CCL3', 'NFKBIA'],#legend_loc='on data',
    basis='X_tsne',
        vmin='p1',
    vmax='p99',
    frameon=False, ncols=2,
    save=  proj_name+'_marker_genes.png'
)

In [ ]:
sc.pl.violin(pair_mono_adata, ['TNF'], groupby='AIFI_L3', rotation=90, save=proj_name+'TNF_violinplot.png')

In [ ]:
pair_mono_adata

In [ ]:
# save data
pair_mono_adata.write_h5ad(data_path + 'ALTRA_scRNA_monocytes_paired_certPro.h5ad')

# Session Info

In [ ]:
import sinfo
sinfo.sinfo(write_req_file = False)